In [2]:
-- 03_Gold_Daily.sql
-- Production daily Gold load. Requires 03_Gold_Setup to have been run.
SET NOCOUNT ON;

IF OBJECT_ID('dbo.fact_rides','U') IS NULL THROW 50001, 'Gold setup missing. Run 03_Gold_Setup first.', 1;

-- Dimensions first, fact last.
EXEC dbo.usp_load_dim_date;
EXEC dbo.usp_load_dim_passenger;
EXEC dbo.usp_load_dim_driver;
EXEC dbo.usp_load_dim_city;
EXEC dbo.usp_load_dim_vehicle_type;
EXEC dbo.usp_load_dim_vehicle_make;
EXEC dbo.usp_load_dim_payment_method;
EXEC dbo.usp_load_dim_ride_status;
EXEC dbo.usp_load_dim_cancellation_reason;
EXEC dbo.usp_load_fact_rides;

-- Lightweight production quality gates. Fail the pipeline if core integrity breaks.
IF EXISTS (SELECT RideID FROM dbo.fact_rides GROUP BY RideID HAVING COUNT(*) > 1)
 THROW 50002, 'Validation failed: duplicate RideID in fact_rides.', 1;
IF EXISTS (SELECT DriverID FROM dbo.dim_driver WHERE IsCurrent=1 GROUP BY DriverID HAVING COUNT(*) > 1)
 THROW 50003, 'Validation failed: multiple current driver versions.', 1;
IF EXISTS (SELECT PassengerID FROM dbo.dim_passenger WHERE IsCurrent=1 GROUP BY PassengerID HAVING COUNT(*) > 1)
 THROW 50004, 'Validation failed: multiple current passenger versions.', 1;
IF EXISTS (SELECT 1 FROM dbo.fact_rides f LEFT JOIN dbo.dim_date d ON f.DateKey=d.DateKey WHERE d.DateKey IS NULL)
 THROW 50005, 'Validation failed: invalid DateKey in fact_rides.', 1;

SELECT COUNT(*) AS FactRideCount, COUNT(DISTINCT RideID) AS UniqueRideCount FROM dbo.fact_rides;